# Notebook 33 — Chronological out-of-fold Random Forest stacking

**Research only.** Compare the full-history Random Forest to cross-fitted probability stacking. The historical 20% final holdout and prospective prediction ledger remain untouched. Every OOF prediction comes from a model trained on strictly earlier completed labels. The outer validation is later than all model training and meta-model fitting.

**Important limitation:** OOF meta models learn from earlier base-model fits while validation probabilities come from a refit on the full outer training window; distribution shift is possible. Evaluate the full-history baseline on the same outer folds.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':ROOT=ROOT.parent
if not (ROOT/'src').exists():raise FileNotFoundError('Open inside stock_ml_dynamic repository')
sys.path.insert(0,str(ROOT/'src'))
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.oof_stacking_research import run_research
SEED=42
N_OUTER=3
OOF_WINDOWS=5
CAP=60000
TREES=200
LEAF=20
CLIP=(.005,.995)
THRESHOLD=.45
COST_BPS=10
OUTPUT=ROOT/'artifacts'/'notebook33_oof_stacking'
print('Root:',ROOT)

## 1. Load the persisted master table

In [ ]:
master,latest,bundle,manifest=load_master(ROOT/'artifacts/master_tables/1d')
print('Master:',master.shape,'Latest:',latest.shape)
print('Date range:',master.bar_timestamp.min(),master.bar_timestamp.max())
print('Tickers:',master.symbol.nunique())

## 2. Run chronological OOF research

This may take time on the workstation. Four methods × three outer folds; five inner OOF base fits per outer fold. No random row shuffling.

In [ ]:
result=run_research(master,n_outer=N_OUTER,windows=OOF_WINDOWS,cap=CAP,trees=TREES,leaf=LEAF,
                    seed=SEED,clip=CLIP,threshold=THRESHOLD,cost_bps=COST_BPS,output_dir=OUTPUT)
outer=result['outer'];audit=result['audit']
print('Development cutoff:',result['cutoff'],'Feature count:',result['feature_count'])
display(outer)

## 3. Audit strictly chronological training and prediction windows

In [ ]:
display(audit)
assert (pd.to_datetime(audit.latest_train_outcome,utc=True)<pd.to_datetime(audit.earliest_pred_bar,utc=True)).all()
print('PASS: all OOF training labels predate prediction bars')

## 4. Compare model performance

In [ ]:
metrics=['accuracy','majority_baseline','auc','logloss','brier_up','coverage','up_precision','mean_net','entry_count']
summary=outer.groupby('model')[metrics].agg(['mean','std']).round(5)
display(summary)
for metric in ['accuracy','auc','logloss','up_precision','mean_net']:
    ax=outer.pivot(index='fold',columns='model',values=metric).plot(kind='bar',figsize=(11,4),title=f'{metric}: outer-fold comparison')
    ax.set_xlabel('Chronological outer fold');ax.grid(axis='y',alpha=.2);plt.tight_layout();plt.show()

## 5. Paired differences against full-history Random Forest

In [ ]:
base=outer.loc[outer.model.eq('full_history_rf')].set_index('fold')
paired=[]
for name,g in outer.groupby('model'):
    if name=='full_history_rf':continue
    g=g.set_index('fold')
    for metric in ['accuracy','auc','logloss','up_precision','mean_net','coverage']:
        diff=(g[metric]-base[metric]).dropna()
        paired.append({'model':name,'metric':metric,'mean_difference':diff.mean(),'min_difference':diff.min(),
                       'max_difference':diff.max(),'positive_folds':int((diff>0).sum()),'folds':len(diff)})
display(pd.DataFrame(paired).round(5))

## 6. Interpretation and decision rules

A stacking method is promising only if it improves outer-fold discrimination or calibration consistently, retains practical entry coverage, and does not concentrate apparent gains into a few trades. A positive mean return alone is insufficient; these diagnostics do not model slippage, portfolio constraints, or correlated trades. Do not promote a model from these development-fold results alone.

In [ ]:
print('Saved outputs to:',OUTPUT)
print('Best mean accuracy:',outer.groupby('model').accuracy.mean().sort_values(ascending=False).head())
print('No historical final-holdout evaluation performed.')